# FX.fact_ohlc — Simple CRUD Test

Insert, read, update, and delete a row from the existing `FX.fact_ohlc` table.

In [1]:
from datetime import datetime, timezone
from decimal import Decimal

from sqlalchemy import text

from imdr.config.settings import get_settings
from imdr.connectors.mssql import MSSQLConnector
get_settings.cache_clear()
connector = MSSQLConnector(get_settings())
print("Connected:", connector.engine.url)

Connected: mssql+pyodbc://@rv-database-1.ctym72ljvrjq.ap-southeast-1.rds.amazonaws.com:1433/IMDR?Trusted_Connection=yes&driver=SQL+Server


## 1. CREATE — Insert a row

In [ ]:
insert_sql = text("""
    INSERT INTO FX.fact_ohlc
        (ts, symbol, series, tenor, deal_type, pair_used,
         open_px, high_px, low_px, close_px, mid_px,
         mid_mean_px, mid_median_px, bid, ask, n_ticks)
    OUTPUT INSERTED.id
    VALUES
        (:ts, :symbol, :series, :tenor, :deal_type, :pair_used,
         :open_px, :high_px, :low_px, :close_px, :mid_px,
         :mid_mean_px, :mid_median_px, :bid, :ask, :n_ticks)
""")

params = dict(
    ts=datetime.now(timezone.utc),
    symbol="EURUSD",
    series="spot",
    tenor="ON",
    deal_type="outright",
    pair_used="EURUSD",
    open_px=Decimal("1.08400000"),
    high_px=Decimal("1.08600000"),
    low_px=Decimal("1.08200000"),
    close_px=Decimal("1.08450000"),
    mid_px=Decimal("1.08425000"),
    mid_mean_px=Decimal("1.08410000"),
    mid_median_px=Decimal("1.08420000"),
    bid=Decimal("1.08400000"),
    ask=Decimal("1.08450000"),
    n_ticks=150,
)

with connector.session() as session:
    result = session.execute(insert_sql, params)
    inserted_id = result.scalar()

print(f"Inserted row with id = {inserted_id}")

## 2. READ — Fetch the row back

In [3]:
import pandas as pd

read_sql = text("SELECT * FROM FX.dim_currency_pair")

with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection())

print(f"Total rows: {len(df)}")
df.head(10)

Total rows: 17


,id,base_ccy,quote_ccy,ccy_class,created_at,updated_at
0,1,EUR,USD,g10,2026-03-11 15:02:46.9457450 +08:00,2026-03-11 15:02:46.9457450 +08:00
1,2,GBP,USD,g10,2026-03-11 15:02:46.9617409 +08:00,2026-03-11 15:02:46.9617409 +08:00
2,3,USD,JPY,g10,2026-03-11 15:02:46.9767432 +08:00,2026-03-11 15:02:46.9767432 +08:00
3,4,AUD,USD,g10,2026-03-11 15:02:46.9907451 +08:00,2026-03-11 15:02:46.9907451 +08:00
4,5,NZD,USD,g10,2026-03-11 15:02:47.0037424 +08:00,2026-03-11 15:02:47.0037424 +08:00
5,6,USD,CAD,g10,2026-03-11 15:02:47.0177422 +08:00,2026-03-11 15:02:47.0177422 +08:00
6,7,USD,CHF,g10,2026-03-11 15:02:47.0317406 +08:00,2026-03-11 15:02:47.0317406 +08:00
7,8,USD,NOK,g10,2026-03-11 15:02:47.0457436 +08:00,2026-03-11 15:02:47.0457436 +08:00
8,9,USD,SEK,g10,2026-03-11 15:02:47.0587420 +08:00,2026-03-11 15:02:47.0587420 +08:00
9,10,USD,CNH,g10,2026-03-11 15:02:47.0727404 +08:00,2026-03-11 15:02:47.0727404 +08:00


## 3. UPDATE — Change close_px

In [ ]:
update_sql = text("""
    UPDATE FX.fact_ohlc
    SET close_px = :close_px
    WHERE id = :id
""")

with connector.session() as session:
    session.execute(update_sql, {"close_px": Decimal("1.09000000"), "id": inserted_id})

# Verify the update
with connector.session() as session:
    df = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Updated close_px = {df['close_px'].iloc[0]}")
df

## 4. DELETE — Remove the test row

In [ ]:
delete_sql = text("DELETE FROM FX.fact_ohlc")

with connector.session() as session:
    result = session.execute(delete_sql)

print(f"Deleted {result.rowcount} row(s)")

# Verify table is empty
with connector.session() as session:
    df = pd.read_sql(text("SELECT COUNT(*) AS cnt FROM FX.fact_ohlc"), session.connection())

print(f"Rows remaining: {df['cnt'].iloc[0]}")

In [ ]:
connector.dispose()
print("Done — connection pool closed.")